# FASE 4 - Deep Learning (Modelo Secuencial GRU)

## Paso 1: Carga de Datos y Scores Híbridos (ML)
Cargamos el dataset limpio para construir las secuencias históricas y el CSV con los **scores híbridos (70% Colaborativo + 30% Contenido)** exportados desde la fase de Machine Learning (`ml_scores_hibrido.csv`).

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# 1. Cargar el dataset limpio para las secuencias
ruta_dataset_limpio = Path('../data/2020-Apr-L.csv')
df_dl = pd.read_csv(ruta_dataset_limpio, parse_dates=['event_time'])

# 2. Cargar el CSV con el resultado híbrido de la fase ML
ruta_ml_hibrido = Path('../data/ml_scores_hibrido.csv')
df_scores_hibrido = pd.read_csv(ruta_ml_hibrido)

# 3. Mapeo de IDs a índices numéricos para los Embeddings (0 a N-1)
usuarios_unicos = df_dl['user_id'].unique()
productos_unicos = df_dl['product_id'].unique()

user_to_index = {user_id: idx for idx, user_id in enumerate(usuarios_unicos)}
product_to_index = {product_id: idx for idx, product_id in enumerate(productos_unicos)}

df_dl['user_idx'] = df_dl['user_id'].map(user_to_index)
df_dl['product_idx'] = df_dl['product_id'].map(product_to_index)

df_scores_hibrido['user_idx'] = df_scores_hibrido['user_id'].map(user_to_index)
df_scores_hibrido['product_idx'] = df_scores_hibrido['product_id'].map(product_to_index)

# 4. Definición de variables de entorno reales
NUM_USUARIOS = len(usuarios_unicos)
NUM_PRODUCTOS = len(productos_unicos)
LONGITUD_SECUENCIA = 10
EMBEDDING_DIM = 64

print("---- UNIFICACIÓN COMPLETADA ----")
print(f"Usuarios (Dimensión User Embedding): {NUM_USUARIOS}")
print(f"Productos (Dimensión Product Embedding): {NUM_PRODUCTOS}")

## Paso 2: Generación de los Tensores de Entrenamiento (Uso del Score Híbrido)
Aquí es donde se **cruza la secuencia del usuario con los resultados híbridos de ML**. 
Por cada usuario, tomamos sus N últimos productos como contexto temporal. Luego evaluamos los candidatos recomendados por la capa ML, usando el `score_ml_refinado` como entrada adicional para el modelo.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Agrupar los historiales de los usuarios ordenados por tiempo
historial_usuarios = df_dl.sort_values('event_time').groupby('user_idx')['product_idx'].apply(list).to_dict()

X_secuencias = []
X_usuarios = []
X_candidatos = []
X_ml_scores = []
y_targets = []

# Iteramos sobre las recomendaciones híbridas pre-calculadas en ML
for index, row in df_scores_hibrido.dropna(subset=['user_idx', 'product_idx']).iterrows():
    uid = int(row['user_idx'])
    candidato_id = int(row['product_idx'])
    ml_score = row['score_ml_refinado']
    
    # Obtenemos el historial del usuario (excluyendo el candidato actual si ya existe)
    historial = historial_usuarios.get(uid, [])
    
    X_usuarios.append(uid)
    X_candidatos.append(candidato_id)
    X_ml_scores.append(ml_score)
    X_secuencias.append(historial[-LONGITUD_SECUENCIA:]) # Tomamos los últimos N productos
    
    # Para entrenar la capa DL, el target suele ser si realmente hubo interacción o no.
    # Si el candidato está en su historial real, es 1. (Asumiendo validación retrospectiva)
    target = 1 if candidato_id in historial else 0 
    y_targets.append(target)

# Convertimos a tensores numpy listos para Keras
X_secuencias = pad_sequences(X_secuencias, maxlen=LONGITUD_SECUENCIA, padding='pre')
X_usuarios = np.array(X_usuarios)
X_candidatos = np.array(X_candidatos)
X_ml_scores = np.array(X_ml_scores)
y_targets = np.array(y_targets)

print(f"Tensores generados: {len(X_usuarios)} muestras listas para entrenar el modelo GRU.")

## Paso 3: Definición de Entradas del Modelo
Ahora sí, con los datos preparados (tensores `X`), definimos las puertas de entrada (Inputs) de la arquitectura Keras.

In [ ]:
from tensorflow.keras.layers import Input

input_secuencia = Input(shape=(LONGITUD_SECUENCIA,), name='secuencia_productos')
input_usuario = Input(shape=(1,), name='usuario_id')
input_candidato = Input(shape=(1,), name='producto_candidato')
input_ml_score = Input(shape=(1,), name='ml_score')

## Paso 4: Capas de Embedding y Red GRU
Transformamos los índices en vectores densos y pasamos la secuencia temporal por la red recurrente GRU.

In [ ]:
from tensorflow.keras.layers import Embedding, GRU, Flatten

emb_producto_layer = Embedding(input_dim=NUM_PRODUCTOS, output_dim=EMBEDDING_DIM, name="emb_producto")
emb_usuario_layer = Embedding(input_dim=NUM_USUARIOS, output_dim=EMBEDDING_DIM, name="emb_usuario")

emb_secuencia = emb_producto_layer(input_secuencia)
emb_usuario = Flatten()(emb_usuario_layer(input_usuario))
emb_candidato = Flatten()(emb_producto_layer(input_candidato))

# Capa GRU que captura el comportamiento secuencial (Vector DL)
vector_dl = GRU(units=64, name="Capa_GRU")(emb_secuencia)

## Paso 5: Fusión de Contextos (Predicción Final)
Concatenamos el Vector DL (comportamiento), el perfil del usuario, el producto candidato y la afinidad calculada en ML.

In [ ]:
from tensorflow.keras.layers import Concatenate, Dense, Dropout

caracteristicas_fusionadas = Concatenate(name="fusion_caracteristicas")([
    vector_dl,       # Información secuencial
    emb_usuario,     # Rasgos generales del usuario
    emb_candidato,   # Rasgos del producto
    input_ml_score   # Peso proveniente del Score Híbrido
])

x = Dense(128, activation='relu')(caracteristicas_fusionadas)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
output_score = Dense(1, activation='sigmoid', name="score_final")(x)

## Paso 6: Compilación y Entrenamiento del Modelo
Ensamblamos el modelo y lo entrenamos usando los tensores de datos que preparamos en el Paso 2.

In [ ]:
from tensorflow.keras.models import Model

modelo_dl = Model(
    inputs=[input_secuencia, input_usuario, input_candidato, input_ml_score],
    outputs=output_score,
    name="UrbanSoul_Hybrid_GRU"
)

modelo_dl.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

modelo_dl.summary()

"""
# Código para entrenar el modelo (Descomentar para ejecutar)
historial = modelo_dl.fit(
    x=[
        X_secuencias, 
        X_usuarios, 
        X_candidatos, 
        X_ml_scores  # <--- AQUÍ entra físicamente el score híbrido de ML
    ],
    y=y_targets,
    epochs=5,
    batch_size=256,
    validation_split=0.2
)
"""